# Infer-18 — Détection de Rupture (Change-Point) : inférer le moment d'un changement de régime

> **Corpus bayésien Infer.NET.** Ce notebook (18ᵉ du corpus) complète la famille des
> « séquences dans le temps » : il étend [Infer-11 (Séquences / HMM)](Infer-11-Sequences.ipynb)
> et [Infer-17 (Filtre de Kalman)](Infer-17-Kalman-Filter.ipynb) au cas où ce n'est pas l'état
> qui change à chaque instant, mais la **structure** du processus qui bascule **une seule fois**
> à un instant inconnu.

**Durée estimée** : 50 minutes
**Prérequis** : [Infer-11-Sequences](Infer-11-Sequences.ipynb) (HMM, `Variable.ForEach`), [Infer-2-Gaussian-Mixtures](Infer-2-Gaussian-Mixtures.ipynb) (conjugaison gaussienne), [Infer-10-Model-Selection](Infer-10-Model-Selection.ipynb) (évidence, Bayes factors)

Imaginez une chaîne de production dont la qualité dérive soudainement, un marché financier
qui change de régime, ou le taux d'accidents miniers qui chute après une nouvelle réglementation.
Dans tous ces cas, le signal observé suit un régime stable, puis **bascule** vers un autre régime
à un instant `cp` que l'on ne connaît pas à l'avance. La question n'est pas « quel est l'état à
chaque pas ? » (répondue par le HMM ou le filtre de Kalman), mais « **à quel instant unique** le
processus a-t-il changé de nature ? ». C'est l'inférence d'un **point de rupture** (*change-point*).

Ce notebook montre comment Infer.NET traite ce problème de façon native : un *a priori* uniforme
sur l'indice de rupture (`Variable.DiscreteUniform`), une sélection de vraisemblance conditionnée
par cet indice (`Variable.If` / `Variable.IfNot` sur une plage), et le moteur **EP** qui retourne
un postérieur **discret** sur la localisation de la rupture — l’idiome message-passing-sur-graphe
de-facteurs que Infer.NET est conçu pour résoudre.

## 1. Motivation : là où le HMM et le filtre de Kalman s'arrêtent

[Infer-11](Infer-11-Sequences.ipynb) modélise une séquence où l'état caché est **discret** et
change à **chaque instant** (météo d'aujourd'hui, mot de la phrase). [Infer-17](Infer-17-Kalman-Filter.ipynb)
est son pendant **continu** : l'état (position, prix) évolue pas-à-pas. Les deux infèrent un état
**récurrent** — un par pas de temps.

Le **point de rupture** est différent : l'inconnue n'est pas une trajectoire d'états, mais **un
unique entier** `cp ∈ {0, …, N−1}` — l'indice où le processus bascule. On veut le postérieur
$p(\text{cp} \mid y_{0:N-1})$, une distribution discrète sur $N$ positions.

L'idiome Infer.NET repose sur trois briques que les notebooks précédents n'exploitent **pas**
ensemble :

| Brique | Rôle | Où déjà vu ? |
|--------|------|--------------|
| `Variable.DiscreteUniform(N)` | *a priori* uniforme sur l'indice de rupture | Infer-3 (Monty Hall) |
| `Variable.ForEach` sur la plage temporelle | une vraisemblance par instant | Infer-11, Infer-17 |
| `Variable.If(block.Index <= cp)` / `IfNot` | sélectionner la vraisemblance *avant*/*après* la rupture | Infer-3 (If/IfNot statique) |

La nouveauté est le couplage : la condition `block.Index <= cp` (l'indice de boucle, obtenu via
`ForEachBlock.Index`) **branche la vraisemblance sur un indice latent couplé à toute la plage** —
un usage plus riche du `If` que le cas statique de Infer-3. EP propage les messages et retourne
une `Discrete` (un vecteur de probabilités sur les $N$ positions) pour `cp`, plus les postérieurs
des paramètres de chaque segment.

In [1]:
#r "nuget: Microsoft.ML.Probabilistic"
#r "nuget: Microsoft.ML.Probabilistic.Compiler"
// restore Infer.NET -- isole dans sa propre cellule (convention de la serie)

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.ML.Probabilistic, 0.4.2504.701 Microsoft.ML.Probabilistic.Compiler, 0.4.2504.701

Configuration des espaces de noms Infer.NET. Le moteur **EP** (Expectation Propagation) est
celui qui traite les facteurs `If` discrets couplés à une plage. On définit aussi le helper
de tirage gaussien (Box-Muller) pour générer une série synthétique dont on connaît le vrai
point de rupture.

In [2]:
using Microsoft.ML.Probabilistic;
using Microsoft.ML.Probabilistic.Distributions;
using Microsoft.ML.Probabilistic.Models;
using Microsoft.ML.Probabilistic.Algorithms;
using Range = Microsoft.ML.Probabilistic.Models.Range;   // desambiguise vs System.Range
using System;
using System.Linq;

// Helper : tirage N(0, 1) par Box-Muller (defini AVANT usage).
double Randn(Random r) {
    double u1 = 1.0 - r.NextDouble();
    double u2 = 1.0 - r.NextDouble();
    return Math.Sqrt(-2.0 * Math.Log(u1)) * Math.Sin(2.0 * Math.PI * u2);
}

// --- Terrain de jeu : serie avec un SEUL changement de regime ---
// Le vrai point de rupture est CACHE ; on verifiera que le modele le recupere.
Random rng = new Random(42);
int N = 100;
int vraiCp = 50;                  // vrai indice de rupture (inconnu du modele)
double muAvant = 2.0, muApres = 7.0;   // moyennes des deux regimes
double sigma = 1.5;               // bruit d'observation
double[] data = new double[N];
for (int i = 0; i < N; i++) {
    double mu = (i <= vraiCp) ? muAvant : muApres;
    data[i] = mu + Randn(rng) * sigma;
}
Console.WriteLine($"Serie generee : N={N} points, vrai cp cache a t={vraiCp} (mu {muAvant} -> {muApres}, sigma={sigma})");

Serie generee : N=100 points, vrai cp cache a t=50 (mu 2 -> 7, sigma=1,5)


## 2. Le modèle de point de rupture gaussien

On déclare `cp` comme un entier d'*a priori* uniforme sur `{0, …, N−1}`, deux moyennes de segment
(*a priori* gaussiens vagues) et une précision commune. Pour chaque instant `t`, on **branche** la
vraisemblance selon la position de `cp` : gaussienne de moyenne `mean1` avant la rupture, `mean2`
après. C'est le couplage `Variable.ForEach` + `If(block.Index <= cp)` qui fait tout le travail
(l'indice de boucle est obtenu via `block.Index`, l'API moderne d'Infer.NET).

EP retourne : un postérieur **`Discrete`** sur `cp` (vecteur de $N$ probabilités), et les postérieurs
gaussiens/gamma des paramètres. Notons que **rien n'est échantillonné** : c'est de l'inférence
déterministe par passage de messages, en quelques itérations.

In [3]:
// --- Modele de change-point gaussien ---
Range t = new Range(N);
VariableArray<double> y = Variable.Array<double>(t);
y.ObservedValue = data;

Variable<int> cp = Variable.DiscreteUniform(N);                      // localisation de la rupture
Variable<double> mean1 = Variable.GaussianFromMeanAndVariance(0.0, 100.0);
Variable<double> mean2 = Variable.GaussianFromMeanAndVariance(0.0, 100.0);
Variable<double> precision = Variable.GammaFromMeanAndVariance(1.0, 1.0);

using (var block = Variable.ForEach(t)) {
    using (Variable.If(block.Index <= cp)) {                         // segment AVANT
        y[t] = Variable.GaussianFromMeanAndPrecision(mean1, precision);
    }
    using (Variable.IfNot(block.Index <= cp)) {                      // segment APRES
        y[t] = Variable.GaussianFromMeanAndPrecision(mean2, precision);
    }
}

var ie = new InferenceEngine(new ExpectationPropagation()) { ShowProgress = false, NumberOfIterations = 50 };
Discrete cpPost = ie.Infer<Discrete>(cp);
Gaussian m1Post = ie.Infer<Gaussian>(mean1);
Gaussian m2Post = ie.Infer<Gaussian>(mean2);
Gamma precPost  = ie.Infer<Gamma>(precision);

Console.WriteLine("=== Posterieur sur le point de rupture cp ===");
Console.WriteLine($"  mode (argmax)        = {cpPost.GetMode()}    (vrai cp cache = {vraiCp})");
Console.WriteLine($"  moyenne              = {cpPost.GetMean(),6:F2}");
Console.WriteLine($"Moyenne avant  (mu1) : {m1Post.GetMean(),6:F3}  (vrai {muAvant})");
Console.WriteLine($"Moyenne apres  (mu2) : {m2Post.GetMean(),6:F3}  (vrai {muApres})");
Console.WriteLine($"Precision commune    : {precPost.GetMean(),6:F3}  (vrai {1.0/(sigma*sigma):F3})");

=== Posterieur sur le point de rupture cp ===


  mode (argmax)        = 50    (vrai cp cache = 50)


  moyenne              =  50,00


Moyenne avant  (mu1) :  1,892  (vrai 2)


Moyenne apres  (mu2) :  7,197  (vrai 7)


Precision commune    :  0,550  (vrai 0,444)


**Lecture.** Le mode du postérieur sur `cp` doit tomber **très près** de 50 (le vrai point caché),
et les moyennes de segment doivent approcher 2 et 7. Si le signal est fort (`|mu2 − mu1| ≫ sigma`),
le postérieur se concentre sur **quelques indices** ; s'il est faible, il s'étale. Examinons la
forme exacte de ce postérieur discret.

In [4]:
// --- Visualisation : postérieur discret sur la localisation de la rupture ---
double pmax = 0.0;
for (int i = 0; i < N; i++) pmax = Math.Max(pmax, cpPost[i]);
Console.WriteLine("Postérieur sur cp (masses > 0.1%, normalisees au pic) :");
for (int i = 0; i < N; i++) {
    if (cpPost[i] < 0.001) continue;                                // on n'affiche que la masse utile
    int barLen = (int)Math.Round(cpPost[i] / pmax * 50.0);
    Console.WriteLine($"  t={i,3}  {new string('#', barLen),-50}  {cpPost[i],7:F4}");
}

Postérieur sur cp (masses > 0.1%, normalisees au pic) :


  t= 50  ##################################################   0,9978


  t= 51                                                       0,0022


## 3. Le cas canonique : les catastrophes minières (loi de Poisson)

Le jeu de données historique du change-point bayésien est la série annuelle des **catastrophes
de mines de charbon** en Grande-Bretagne (1851–1962, Jarrett 1979). Il s'agit de **comptes** :
le modèle naturel est une loi de **Poisson** dont le *taux* bascule une fois — de ~3 accidents/an
avant la réforme de sécurité des années 1880 à ~1/an après.

Même idiome que le cas gaussien : `DiscreteUniform` sur l'année de rupture, `If/IfNot` sur la
plage des années, mais `Variable.Poisson(taux)` en lieu et place de la gaussienne. EP retourne le
postérieur `Discrete` sur l'année de rupture et les taux `Gamma` des deux régimes.

In [5]:
// --- Catastrophes minieres britanniques, 1851-1962 (Jarrett 1979) ---
int[] disasters = new int[] {
    4,5,4,0,1,4,3,4,0,6, 3,3,4,0,2,6,3,3,5,4,
    5,3,1,4,4,1,5,5,3,4, 2,5,2,2,3,4,2,1,3,2,
    2,1,1,1,1,3,0,0,1,0, 1,1,0,0,3,1,0,3,2,2,
    0,1,1,1,0,1,0,1,0,0, 0,2,1,0,0,0,1,1,0,2,
    3,3,1,1,2,1,1,1,1,2, 4,2,0,0,1,4,0,0,0,1,
    0,0,0,0,1,0,0,1,0,1, 0,1
};
int N2 = disasters.Length;
Console.WriteLine($"Serie cataclysmes miniers : {N2} annees ({1851}..{1851 + N2 - 1})");

Range an = new Range(N2);
VariableArray<int> d = Variable.Array<int>(an);
d.ObservedValue = disasters;

Variable<int> cp2 = Variable.DiscreteUniform(N2);
Variable<double> tauxAvant  = Variable.GammaFromMeanAndVariance(3.0, 100.0);
Variable<double> tauxApres  = Variable.GammaFromMeanAndVariance(1.0, 100.0);

// Initialisation data-driven : EP est deterministe (pas d'echantillonnage), donc un point de
// rupture peut pieger EP dans un optimum local. On amorce les deux taux vers les extremes
// empiriques de la serie (moyennes des premieres et des dernieres annees) -- une heuristique
// standard pour les modeles de change-point sous EP.
double estAvant = disasters.Take(10).Average();
double estApres = disasters.Skip(N2 - 10).Take(10).Average();
tauxAvant.InitialiseTo(Gamma.FromMeanAndVariance(estAvant, 1.0));
tauxApres.InitialiseTo(Gamma.FromMeanAndVariance(estApres, 1.0));

using (var block2 = Variable.ForEach(an)) {
    using (Variable.If(block2.Index <= cp2)) {
        d[an] = Variable.Poisson(tauxAvant);
    }
    using (Variable.IfNot(block2.Index <= cp2)) {
        d[an] = Variable.Poisson(tauxApres);
    }
}

var ie2 = new InferenceEngine(new ExpectationPropagation()) { ShowProgress = false, NumberOfIterations = 50 };
Discrete cp2Post = ie2.Infer<Discrete>(cp2);
Gamma tauxAvantPost = ie2.Infer<Gamma>(tauxAvant);
Gamma tauxApresPost = ie2.Infer<Gamma>(tauxApres);

int modeCp2 = cp2Post.GetMode();
Console.WriteLine($"Annee de rupture detectee : {1851 + modeCp2}  (indice {modeCp2})");
Console.WriteLine($"Taux AVANT  : {tauxAvantPost.GetMean(),5:F2} catastrophes/an");
Console.WriteLine($"Taux APRES  : {tauxApresPost.GetMean(),5:F2} catastrophes/an");
Console.WriteLine($"Rapport de taux : {tauxAvantPost.GetMean() / Math.Max(1e-6, tauxApresPost.GetMean()):F1}x");

Serie cataclysmes miniers : 112 annees (1851..1962)


Annee de rupture detectee : 1891  (indice 40)


Taux AVANT  :  3,12 catastrophes/an


Taux APRES  :  0,94 catastrophes/an


Rapport de taux : 3,3x


**Lecture.** Le postérieur concentre la rupture autour de **1890–1891** — soit exactement
la période des grandes réformes de sécurité dans les mines britanniques. Le taux passe de ~3,1 à
~0,9 catastrophe par an, un rapport de ~3,3×. C'est le résultat historique de la littérature
sur les modèles de point de rupture (Carlin, Gelfand & Smith 1992 ; PyMC en fait son exemple
introductif). **Aucun échantillonnage MCMC** n'est utilisé ici : EP propage les messages sur le
graphe de facteurs et renvoie le postérieur en quelques itérations.

## 4. Vraie rupture ou caprice du bruit ? Le piège de la flexibilité

Un modèle de point de rupture est **flexible** : il peut, si on le laisse faire, trouver une
« meilleure » coupure dans n'importe quelle série — y compris du pur bruit. C'est le piège du
surajustement. Pour le mesurer, on génère une série **sans rupture** (même moyenne du début à la
fin) et on lui applique le **même** modèle. On quantifie la concentration du postérieur sur `cp`
par son **entropie** $H$ (en bits) : $H_{\max} = \log_2 N$ pour un postérieur plat (aucune idée sur
la localisation), $H \to 0$ pour un postérieur piqué (localisation quasi certaine).

Le résultat est instructif : une **vraie** rupture fait chuter l'entropie vers 0, mais même du
pur bruit laisse l'entropie **nettement sous** le maximum — le modèle surajuste et « trouve » une
coupure spurieuse. Distinguer les deux exige donc plus que la seule concentration : un **Bayes
factor** (rapport de vraisemblance marginale) qui pénalise la flexibilité supplémentaire du modèle
à rupture (rasoir d'Occam) — c'est l'objet d'[Infer-10](Infer-10-Model-Selection.ipynb).

In [6]:
// Entropie (bits) du postérieur de cp sur la serie AVEC rupture
double Hcp = 0.0, HmaxReel = Math.Log(N, 2);
for (int i = 0; i < N; i++) { double p = cpPost[i]; if (p > 1e-12) Hcp -= p * Math.Log(p, 2); }

// --- Controle : serie SANS rupture (moyenne constante) ---
Random rng2 = new Random(7);
int Nc = 100;
double[] dataCtrl = new double[Nc];
double muConst = 5.0;
for (int i = 0; i < Nc; i++) dataCtrl[i] = muConst + Randn(rng2) * 1.5;

Range tc = new Range(Nc);
VariableArray<double> yc = Variable.Array<double>(tc);
yc.ObservedValue = dataCtrl;
Variable<int> cpC = Variable.DiscreteUniform(Nc);
Variable<double> mc1 = Variable.GaussianFromMeanAndVariance(0.0, 100.0);
Variable<double> mc2 = Variable.GaussianFromMeanAndVariance(0.0, 100.0);
Variable<double> pc  = Variable.GammaFromMeanAndVariance(1.0, 1.0);
using (var blockC = Variable.ForEach(tc)) {
    using (Variable.If(blockC.Index <= cpC)) yc[tc] = Variable.GaussianFromMeanAndPrecision(mc1, pc);
    using (Variable.IfNot(blockC.Index <= cpC)) yc[tc] = Variable.GaussianFromMeanAndPrecision(mc2, pc);
}
var ieC = new InferenceEngine(new ExpectationPropagation()) { ShowProgress = false, NumberOfIterations = 50 };
Discrete cpCPost = ieC.Infer<Discrete>(cpC);

double Hctrl = 0.0;
for (int i = 0; i < Nc; i++) { double p = cpCPost[i]; if (p > 1e-12) Hctrl -= p * Math.Log(p, 2); }
double HmaxCtrl = Math.Log(Nc, 2);

Console.WriteLine($"Serie AVEC rupture   : H(cp) = {Hcp,6:F3} bits / {HmaxReel:F3}  =>  information = {HmaxReel - Hcp,5:F3} bits");
Console.WriteLine($"Controle SANS rupture : H(cp) = {Hctrl,6:F3} bits / {HmaxCtrl:F3}  =>  information = {HmaxCtrl - Hctrl,5:F3} bits");
Console.WriteLine();
Console.WriteLine("=> Vraie rupture : H -> 0 (localisation quasi certaine).");
Console.WriteLine("=> Pur bruit    : H reste BAS mais non nul -- le modele SURAJUSTE une coupure spurieuse.");
Console.WriteLine("=> Test decisif (vraie rupture vs bruit) : Bayes factor (cf. Infer-8), pas la seule entropie.");

Serie AVEC rupture   : H(cp) =  0,023 bits / 6,644  =>  information = 6,621 bits


Controle SANS rupture : H(cp) =  0,686 bits / 6,644  =>  information = 5,957 bits


=> Vraie rupture : H -> 0 (localisation quasi certaine).


=> Pur bruit    : H reste BAS mais non nul -- le modele SURAJUSTE une coupure spurieuse.


=> Test decisif (vraie rupture vs bruit) : Bayes factor (cf. Infer-8), pas la seule entropie.


**Lecture.** La série **avec** rupture pousse l'entropie vers 0 (localisation quasi certaine),
tandis que le **contrôle** (pur bruit) laisse une entropie faible mais non nulle (~0,7 bits) :
le modèle, trop flexible, a « trouvé » une coupure spurieuse. C'est le piège classique des
modèles de point de rupture.

La leçon : la concentration du postérieur signale qu'une coupure est **possible**, mais seule la
comparaison de modèles par **Bayes factor** (vraisemblance marginale) tranche entre « vraie
rupture » et « surajustement ». Le Bayes factor pénalise la flexibilité supplémentaire du modèle
à rupture (rasoir d'Occam) : sur la vraie série, le gain de vraisemblance compense largement la
pénalité ; sur le bruit, non. C'est exactement le mécanisme étudié dans
[Infer-10 (Sélection de modèles)](Infer-10-Model-Selection.ipynb).

## 5. Exercices

Les trois exercices ci-dessous exploitent le même idiome `DiscreteUniform` + `If/IfNot` sur une
plage. Ils sont laissés **à compléter** (stubbs sans erreur) — le notebook s'exécute de bout en
bout même non rempli.

### Exercice 1 — Deux points de rupture (trois régimes)

Étendez le modèle à **deux** ruptures `cp1 < cp2` délimitant trois segments de moyennes
`mu1, mu2, mu3`. Le postérieur devient joint sur `(cp1, cp2)`.

**Indice :** générez une série à trois régimes, puis emboîtez les `If` — `If(t <= cp1)` pour le
premier segment, sinon `If(t <= cp2)` pour le second, sinon le troisième.

**Étape 1.** Générez `data3` avec `mu = 2` sur `[0, cp1]`, `mu = 5` sur `[cp1, cp2]`, `mu = 8`
au-delà.
**Étape 2.** Déclarez `cp1, cp2` (`DiscreteUniform`, avec `cp1 < cp2`), trois moyennes, et
emboîtez les `If/IfNot`.
**Étape 3.** Inférez `cp1Post` et `cp2Post` et vérifiez la récupération des deux positions.

In [7]:
// Exercice 1 : deux points de rupture (a completer)
// TODO etudiant : declarer cp1, cp2, mu1, mu2, mu3 et emboiter les If/IfNot sur les trois segments.
Console.WriteLine("Exercice 1 a completer : deux points de rupture (trois regimes).");

Exercice 1 a completer : deux points de rupture (trois regimes).


### Exercice 2 — Un changement de variance (moyenne constante)

Jusqu'ici la rupture portait sur la **moyenne**. Détectez une rupture de **variance** seule : la
moyenne reste constante, mais la précision bascule de `prec1` (régime calme) à `prec2` (régime
turbulent) à l'instant `cp`.

**Indice :** la moyenne `mu` est commune aux deux segments ; seules les précisions diffèrent.
Placez `mu` *en dehors* des `If`, et branchez la précision à l'intérieur.

**Étape 1.** Générez une série de moyenne 5 constante, de variance 0.25 avant `cp` et 4.0 après.
**Étape 2.** Adaptez le modèle (une moyenne commune, deux précisions branchées par `If/IfNot`).
**Étape 3.** Le postérieur sur `cp` se concentre-t-il malgré l'absence de saut de moyenne ?

In [8]:
// Exercice 2 : changement de variance (a completer)
// TODO etudiant : moyenne commune, deux precisions (prec1, prec2) branchees par If/IfNot.
Console.WriteLine("Exercice 2 a completer : detection d'un changement de variance.");

Exercice 2 a completer : detection d'un changement de variance.


### Exercice 3 — Robustesse au signal et à la taille

Étudiez comment la **concentration** du postérieur sur `cp` dépend (a) de l'amplitude du saut
`delta = mu2 − mu1` et (b) de la longueur `N` de la série.

**Indice :** reprenez le modèle de la section 2 dans une boucle sur `delta ∈ {1, 2, 4}` et notez
`H(cp)` à chaque fois.

**Étape 1.** Pour chaque `delta`, générez une série et inférez `cpPost`.
**Étape 2.** Calculez l'entropie `H(cp)` et l'information `Hmax − H`.
**Étape 3.** Observez : un signal faible ou une courte série élargit le postérieur ; un signal
fort ou une longue série le resserre. Quantifiez la transition.

In [9]:
// Exercice 3 : robustesse au signal et a N (a completer)
// TODO etudiant : boucler sur delta dans {1, 2, 4}, mesurer H(cp) a chaque fois.
Console.WriteLine("Exercice 3 a completer : robustesse du posterieur au signal et a la taille.");

Exercice 3 a completer : robustesse du posterieur au signal et a la taille.


## 6. Conclusion

Le **point de rupture** complète la famille des séquences temporelles du corpus :

| Notebook | État caché | Inférence |
|----------|-----------|-----------|
| [Infer-11](Infer-11-Sequences.ipynb) (HMM) | discret, **récurrent** (un par pas) | postérieur sur la trajectoire d'états |
| [Infer-17](Infer-17-Kalman-Filter.ipynb) (Kalman) | continu, **récurrent** | postérieur gaussien pas-à-pas |
| **Infer-18 (change-point)** | **entier structurel unique** | postérieur `Discrete` sur la localisation |

Le trait distinctif : l'inconnue est un **indice structurel** couplé à toute la plage temporelle,
et EP retourne sa distribution discrète — un usage du `Variable.If(t.Item <= cp)` sur une plage
que n'exploite aucun autre notebook de l'arc. C'est aussi le chaînon naturel vers
[Infer-14 (Causal)](Infer-14-Causal-Inference.ipynb) : un point de rupture est un changement de
**mécanisme générateur**, exactement ce que l'inférence causale cherche à détecter (rupture dans
$p(Y \mid do(X))$), et vers la sélection de modèles ([Infer-10](Infer-10-Model-Selection.ipynb)),
où la concentration du postérieur joue le rôle d'un Bayes factor.

**Pour aller plus loin.** Plusieurs ruptures (exercice 1) ; ruptures de variance (exercice 2) ;
nombre de ruptures inconnu (processus de Dirichlet — non natif en Infer.NET, voir le versant PyMC) ;
ou la détection en ligne (filtering) plutôt qu'off-line.